In [1]:
import numpy as np
from pathlib import Path
import geopandas as gpd
from shapely.geometry import Point

In [2]:
path_base_data = Path.cwd() / "data" / "preprocessed prediction data"
path_base_pred = Path.cwd() / "modelT" / "results_pred"

# Reconstructing full image and save results

In [3]:
def reconstruct_from_patches(predictions,positions,full_shape,patch_size=256,stride=128):
    # Reconstruct the full image from patches by averaging overlapping values

    H, W = full_shape

    output = np.zeros((H, W), dtype=np.float32)
    counts = np.zeros((H, W), dtype=np.float32)

    # Go through each pixel
    for pred, (i, j) in zip(predictions, positions):

        pred = pred.squeeze()

        # Get the sum for each pixel and the amount of predictions per pixel
        output[i:i+patch_size, j:j+patch_size] += pred
        counts[i:i+patch_size, j:j+patch_size] += 1

    # Get the average per pixel
    output /= np.maximum(counts, 1)

    return output

In [4]:
def raster_to_tree_geojson(pred_map, ref_profile, out_file=None, mode="center"):
    # Deterministic conversion of raster tree density map to GeoJSON points.#

    transform = ref_profile["transform"]
    crs = ref_profile["crs"]
    H, W = pred_map.shape

    geometries = []
    longitudes = []
    latitudes = []

    # Go through each pixel 
    for i in range(H):
        for j in range(W):
            
            # Get the pixel value and round it
            val = pred_map[i, j]
            n = int(val)

            # If the pixel value is smaller or equal 0 continue
            if n <= 0:
                continue

            # Get pixel bounds in world coordinates
            x0, y0 = transform * (j, i)
            x1, y1 = transform * (j + 1, i + 1)

            # Set all tree locations to pixel center
            if mode == "center":

                x = (x0 + x1) / 2
                y = (y0 + y1) / 2

                # Create an entry for each tree at that pixel
                for _ in range(n):
                    geometries.append(Point(x, y))
                    longitudes.append(x)
                    latitudes.append(y)

            # Spread trees evenly within the pixel
            elif mode == "grid":

                side = int(np.ceil(np.sqrt(n)))

                xs = np.linspace(x0, x1, side + 1)[:-1] + (x1 - x0) / (2 * side)
                ys = np.linspace(y0, y1, side + 1)[:-1] + (y1 - y0) / (2 * side)

                coords = [(xs[k % side], ys[k // side]) for k in range(n)]

                # Create an entry for each tree at its specified location within the pixel
                for x, y in coords:
                    geometries.append(Point(x, y))
                    longitudes.append(x)
                    latitudes.append(y)

    gdf = gpd.GeoDataFrame({"longitude": longitudes, "latitude": latitudes}, geometry=geometries, crs=crs)

    if out_file is not None:
        gdf.to_file(out_file, driver="GeoJSON")

    return gdf

## Athens

In [5]:
# Load data from preprocessing
imgs_ath = np.load(path_base_data / "satellite_patches_ath.npy")
pos_ath = np.load(path_base_data / "positions_patches_ath.npy", allow_pickle=True)
shape_ath = np.load(path_base_data / "shape_patches_ath.npy")
ref_ath = np.load(path_base_data / "reference_patches_ath.npy", allow_pickle=True).item()

# Load predictions
preds_ath = np.load(path_base_pred / "athens" / "Fold_1" / "predictions.npy")
probs_ath = np.load(path_base_pred / "athens" / "Fold_1" / "probabilities.npy")

In [6]:
all_preds_ath = reconstruct_from_patches(preds_ath,pos_ath,shape_ath)
result_path = path_base_pred / "trees_athens.json"
raster_to_tree_geojson(all_preds_ath, ref_ath, out_file=result_path, mode="grid")

,longitude,latitude,geometry
0,-83.478238,34.039547,POINT (-83.47824 34.03955)
1,-83.463416,34.039547,POINT (-83.46342 34.03955)
2,-83.461440,34.039547,POINT (-83.46144 34.03955)
3,-83.457757,34.039547,POINT (-83.45776 34.03955)
4,-83.457667,34.039547,POINT (-83.45767 34.03955)
...,...,...,...
1167536,-83.251616,33.855639,POINT (-83.25162 33.85564)
1167537,-83.251324,33.855662,POINT (-83.25132 33.85566)
1167538,-83.250987,33.855684,POINT (-83.25099 33.85568)
1167539,-83.250942,33.855684,POINT (-83.25094 33.85568)


## Budapest

In [7]:
# Load data from preprocessing
imgs_bud = np.load(path_base_data / "satellite_patches_bud.npy")
pos_bud = np.load(path_base_data / "positions_patches_bud.npy", allow_pickle=True)
shape_bud = np.load(path_base_data / "shape_patches_bud.npy")
ref_bud = np.load(path_base_data / "reference_patches_bud.npy", allow_pickle=True).item()

# Load predictions
preds_bud = np.load(path_base_pred / "budapest" / "Fold_1" / "predictions.npy")
probs_bud = np.load(path_base_pred / "budapest" / "Fold_1" / "probabilities.npy")

In [8]:
all_preds_bud = reconstruct_from_patches(preds_bud,pos_bud,shape_bud)
result_path = path_base_pred / "trees_budapest.json"
raster_to_tree_geojson(all_preds_bud, ref_bud, out_file=result_path, mode="grid")

,longitude,latitude,geometry
0,18.949287,47.613360,POINT (18.94929 47.61336)
1,18.949467,47.613360,POINT (18.94947 47.61336)
2,18.996718,47.613360,POINT (18.99672 47.61336)
3,19.031753,47.613360,POINT (19.03175 47.61336)
4,19.032112,47.613360,POINT (19.03211 47.61336)
...,...,...,...
1836184,19.321190,47.360484,POINT (19.32119 47.36048)
1836185,19.321549,47.360484,POINT (19.32155 47.36048)
1836186,19.321908,47.360484,POINT (19.32191 47.36048)
1836187,19.323256,47.360484,POINT (19.32326 47.36048)


## Milan

In [9]:
# Load data from preprocessing
imgs_mil = np.load(path_base_data / "satellite_patches_mil.npy")
pos_mil = np.load(path_base_data / "positions_patches_mil.npy", allow_pickle=True)
shape_mil = np.load(path_base_data / "shape_patches_mil.npy")
ref_mil = np.load(path_base_data / "reference_patches_mil.npy", allow_pickle=True).item()

# Load predictions
preds_mil = np.load(path_base_pred / "milan" / "Fold_1" / "predictions.npy")
probs_mil = np.load(path_base_pred / "milan" / "Fold_1" / "probabilities.npy")

In [10]:
all_preds_mil = reconstruct_from_patches(preds_mil,pos_mil,shape_mil)
result_path = path_base_pred / "trees_milan.json"
raster_to_tree_geojson(all_preds_mil, ref_mil, out_file=result_path, mode="grid")

,longitude,latitude,geometry
0,9.041678,45.535916,POINT (9.04168 45.53592)
1,9.046439,45.535916,POINT (9.04644 45.53592)
2,9.052188,45.535916,POINT (9.05219 45.53592)
3,9.065933,45.535916,POINT (9.06593 45.53592)
4,9.083270,45.535916,POINT (9.08327 45.53592)
...,...,...,...
585289,9.268323,45.397995,POINT (9.26832 45.39799)
585290,9.268353,45.397995,POINT (9.26835 45.39799)
585291,9.268480,45.398047,POINT (9.26848 45.39805)
585292,9.268525,45.398047,POINT (9.26853 45.39805)


## Prague

In [11]:
# Load data from preprocessing
imgs_pra = np.load(path_base_data / "satellite_patches_pra.npy")
pos_pra = np.load(path_base_data / "positions_patches_pra.npy", allow_pickle=True)
shape_pra = np.load(path_base_data / "shape_patches_pra.npy")
ref_pra = np.load(path_base_data / "reference_patches_pra.npy", allow_pickle=True).item()

# Load predictions
preds_pra = np.load(path_base_pred / "prague" / "Fold_1" / "predictions.npy")
probs_pra = np.load(path_base_pred / "prague" / "Fold_1" / "probabilities.npy")

In [12]:
all_preds_pra = reconstruct_from_patches(preds_pra,pos_pra,shape_pra)
result_path = path_base_pred / "trees_prague.json"
raster_to_tree_geojson(all_preds_pra, ref_pra, out_file=result_path, mode="grid")

,longitude,latitude,geometry
0,14.352428,50.177691,POINT (14.35243 50.17769)
1,14.353035,50.177713,POINT (14.35303 50.17771)
2,14.353079,50.177713,POINT (14.35308 50.17771)
3,14.353147,50.177691,POINT (14.35315 50.17769)
4,14.353237,50.177691,POINT (14.35324 50.17769)
...,...,...,...
1801920,14.695472,49.947834,POINT (14.69547 49.94783)
1801921,14.695517,49.947834,POINT (14.69552 49.94783)
1801922,14.695585,49.947812,POINT (14.69558 49.94781)
1801923,14.695742,49.947834,POINT (14.69574 49.94783)


## Warsaw

In [13]:
# Load data from preprocessing
imgs_war = np.load(path_base_data / "satellite_patches_war.npy")
pos_war = np.load(path_base_data / "positions_patches_war.npy", allow_pickle=True)
shape_war = np.load(path_base_data / "shape_patches_war.npy")
ref_war = np.load(path_base_data / "reference_patches_war.npy", allow_pickle=True).item()

# Load predictions
preds_war = np.load(path_base_pred / "warsaw" / "Fold_1" / "predictions.npy")
probs_war = np.load(path_base_pred / "warsaw" / "Fold_1" / "probabilities.npy")

In [14]:
all_preds_war = reconstruct_from_patches(preds_war,pos_war,shape_war)
result_path = path_base_pred / "trees_warsaw.json"
raster_to_tree_geojson(all_preds_war, ref_war, out_file=result_path, mode="grid")

,longitude,latitude,geometry
0,20.887672,52.368323,POINT (20.88767 52.36832)
1,20.896116,52.368323,POINT (20.89612 52.36832)
2,20.906447,52.368323,POINT (20.90645 52.36832)
3,20.906986,52.368323,POINT (20.90699 52.36832)
4,20.908064,52.368323,POINT (20.90806 52.36832)
...,...,...,...
2336929,21.264538,52.103971,POINT (21.26454 52.10397)
2336930,21.264493,52.103926,POINT (21.26449 52.10393)
2336931,21.264695,52.103948,POINT (21.26469 52.10395)
2336932,21.265211,52.103971,POINT (21.26521 52.10397)
